# Quantum-Selected Configuration Interaction (QSCI)

Quantum-Selected Configuration Interaction (QSCI) [1] is a quantum-classical hybrid algorithm designed for electronic structure calculations, particularly to compute ground state and excited state energies of molecular systems. This method is one of the quantum subspace diagonalization (QSD). It is well-suited for NISQ-era quantum devices, focusing on using quantum resources efficiently while maintaining high accuracy.

![quantum-subspace-diagonalization](images/qsd.png)

QSCI selects this vector space from a sampled computational basis. This notebook compares the original sampled-determinant QSCI subspace with an SBD tensor-product-basis (SBD-TPB) diagonalization built from the sampled alpha and beta determinant sets. The SBD-TPB result should be read as the energy of the alpha/beta product-expanded subspace, not as the energy of exactly the same selected full-determinant subspace used by the original QSCI path.

## 0. Problem definition

Let's define Hamiltonian of $\mathrm{H_4}$.

In [1]:
%pip install --quiet --break-system-packages openfermionpyscf==0.5

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path

import cudaq
import numpy as np
import openfermion
import openfermionpyscf
from openfermion.transforms import get_fermion_operator, jordan_wigner

from qsci_sbd_src import (
    count_alpha_beta_electrons,
    diag_sbd_tpb,
    diagonalize_selected_full_space,
    filter_by_alpha_beta,
    target_alpha_beta_electrons,
)

In [3]:
# Number of hydrogen atoms.
hydrogen_count = 8

# Distance between the atoms in Angstroms.
bond_distance = 1.0

# Define a linear chain of Hydrogen atoms.
geometry = [("H", (0, 0, i * bond_distance)) for i in range(hydrogen_count)]

basis = "sto3g"
multiplicity = 1
charge = 0
shots_count = 10000

molecule = openfermionpyscf.run_pyscf(
    openfermion.MolecularData(geometry, basis, multiplicity, charge), run_fci=True
)
molecular_hamiltonian = molecule.get_molecular_hamiltonian()
fermion_hamiltonian = get_fermion_operator(molecular_hamiltonian)
qubit_hamiltonian = jordan_wigner(fermion_hamiltonian)
qubit_hamiltonian.compress()

hamiltonian = cudaq.SpinOperator(qubit_hamiltonian)

num_qubits = hamiltonian.qubit_count
electron_count = molecule.n_electrons
n_alpha, n_beta = target_alpha_beta_electrons(electron_count, multiplicity)

print(
    f"n_qubits={num_qubits}, n_electrons={electron_count}, "
    f"n_alpha={n_alpha}, n_beta={n_beta}"
)

n_qubits=16, n_electrons=8, n_alpha=4, n_beta=4


## 1. Prepare an Approximate Quantum State

Start with a variational ansatz (e.g., from VQE) to approximate the ground state wavefunction on a quantum computer. For this example, zero-start LBFGS with CUDA-Q forward finite-difference gradients reaches the target accuracy in a small number of iterations.

In [4]:
if cudaq.num_available_gpus() > 0 and cudaq.has_target("nvidia"):
    cudaq.set_target("nvidia")
else:
    print("CUDA or GPU support is unavailable. Running with CPU simulator. Performance may be significantly reduced.")
    cudaq.set_target("qpp-cpu")

In [5]:
@cudaq.kernel
def kernel(thetas: list[float]):
    qubits = cudaq.qvector(num_qubits)

    # Hartree-Fock
    for i in range(electron_count):
        x(qubits[i])

    # UCCSD
    cudaq.kernels.uccsd(qubits, thetas, electron_count, num_qubits)


parameter_count = cudaq.kernels.uccsd_num_parameters(electron_count, num_qubits)


def cost(thetas: list[float]):
    return cudaq.observe(kernel, hamiltonian, thetas).expectation()


# Initial variational parameters. Start from the Hartree-Fock reference.
x0 = np.zeros(parameter_count)

print(f"UCCSD parameters: {parameter_count}")

UCCSD parameters: 360


In [6]:
gradient = cudaq.gradients.ForwardDifference()


def cost_with_gradient(thetas: list[float]):
    exp_val = cost(thetas)
    gradient_vector = gradient.compute(thetas, cost, exp_val)
    return exp_val, gradient_vector

In [7]:
%%time
lbfgs_optimizer = cudaq.optimizers.LBFGS()
lbfgs_optimizer.max_iterations = 6
lbfgs_optimizer.initial_parameters = x0.tolist()

lbfgs_energy, lbfgs_params = lbfgs_optimizer.optimize(
    parameter_count,
    cost_with_gradient,
)

energy, opt_params = lbfgs_energy, lbfgs_params

CPU times: user 6h 56min 16s, sys: 12min 3s, total: 7h 8min 19s
Wall time: 6h 51min 52s


## 2. Quantum Sampling to Select Configurations

Measure this quantum state multiple times to obtain bitstrings, which correspond to Slater determinants (electronic configurations).

In [8]:
sample_result = cudaq.sample(kernel, opt_params, shots_count=shots_count)
print(sample_result)

{ 0000111111001100:1 0000111111110000:1 0001111011000011:1 0001111111100000:1 0010110111100001:1 0011110011001100:1 0011111100000011:2 0011111100001100:3 0011111100010010:3 0011111100100001:2 0011111100110000:4 0011111101000010:1 0011111110000100:2 0011111111000000:11 0100101101001011:1 0100111010110100:1 0101000011111010:1 0101001111100010:1 0101111100101000:1 0101111110100000:1 0110110111001000:1 0110111100000110:1 0110111100001001:6 0110111100011000:1 0110111100100100:1 0110111101000010:4 0110111101100000:7 0110111110000001:4 0110111110010000:17 0110111111000000:2 0111011100101000:1 0111011110000010:1 0111011110001000:2 0111100001001011:1 0111101100100001:8 0111101101001000:7 0111101110000100:3 0111110110000010:6 0111111000000011:1 0111111000001001:2 0111111000100100:1 0111111001000010:4 0111111010000001:11 0111111010000100:1 0111111010010000:5 0111111011000000:2 0111111100000010:2 0111111110000000:16 1000110110001101:1 1001100111110000:1 1001110001100011:1 1001111100000110:1 100111

## 3. Select Determinants

The original QSCI path uses every unique sampled determinant as a full determinant. SBD-TPB (Selected Basis Diagonalization with Tensor Product Basis) first filters the samples to the target electron-number and alpha/beta spin sector, then extracts the unique alpha determinants and unique beta determinants from those filtered samples. The executable used below is from the GPU-accelerated Thrust SBD implementation described in [2].

SBD-TPB diagonalizes the Hamiltonian in the tensor-product space formed by these two sets. If the filtered samples contain alpha set $A$ and beta set $B$, the SBD subspace is $A \times B$. This product space can include full determinants that were not directly sampled, but whose alpha and beta halves were sampled independently. This is why the SBD-TPB energy is compared separately from the original QSCI selected-subspace energy.

The helper functions also keep the spin bookkeeping explicit. CUDA-Q/OpenFermion use the spin-orbital order $\alpha_0, \beta_0, \alpha_1, \beta_1, \ldots$; before calling SBD, the sampled bitstrings are filtered by the target $(N_\alpha, N_\beta)$ sector and converted into the alpha/beta determinant strings expected by the SBD executable.

In [9]:
bitstrs = list(sample_result)
filtered_bitstrs = filter_by_alpha_beta(bitstrs, n_alpha, n_beta)
if not filtered_bitstrs:
    raise ValueError("No sampled determinants matched the target alpha/beta sector.")

sector_counts = {}
for bitstring in bitstrs:
    key = count_alpha_beta_electrons(bitstring)
    sector_counts[key] = sector_counts.get(key, 0) + 1

print(f"Unique sampled determinants: {len(bitstrs)}")
print(f"Target sector: n_alpha={n_alpha}, n_beta={n_beta}")
print("Unique sampled determinant sectors:")
for (sector_alpha, sector_beta), count in sorted(sector_counts.items()):
    print(f"  n_alpha={sector_alpha}, n_beta={sector_beta}: {count}")
print(f"Filtered determinants for SBD-TPB: {len(filtered_bitstrs)}")

Unique sampled determinants: 164
Target sector: n_alpha=4, n_beta=4
Unique sampled determinant sectors:
  n_alpha=4, n_beta=4: 164
Filtered determinants for SBD-TPB: 164


## 4. Classical Diagonalization

Diagonalize the original sampled determinant subspace with the reference Python QSCI implementation, then diagonalize the alpha/beta tensor-product basis with SBD-TPB. The SBD wrapper writes a temporary FCIDUMP file and alpha/beta determinant files, then calls the SBD TPB executable through `mpirun`. The following cells use Jupyter timing magics so the runtime of each diagonalization path is visible in the notebook output.

In [10]:
%%time
original_qsci = diagonalize_selected_full_space(hamiltonian, bitstrs)

CPU times: user 1.59 s, sys: 2.77 ms, total: 1.59 s
Wall time: 1.59 s


In [11]:
%%time
sbd_executable = Path(
    os.environ.get(
        "QSCI_SBD_EXECUTABLE",
        "PATH/TO/sbd/apps/chemistry_tpb_selected_basis_diagonalization/diag",
    )
)
if not sbd_executable.exists():
    raise FileNotFoundError(f"SBD executable not found: {sbd_executable}")

sbd_result = diag_sbd_tpb(
    molecule,
    filtered_bitstrs,
    executable=sbd_executable,
    mpi_processes=int(os.environ.get("QSCI_SBD_MPI_PROCESSES", "1")),
    iteration=8,
    block=20,
    tolerance=1.0e-6,
    verbose=False,
)

CPU times: user 8.35 ms, sys: 2.11 ms, total: 10.5 ms
Wall time: 2.42 s


## 5. Compare results

In [12]:
rows = [
    ("VQE LBFGS", lbfgs_energy),
    ("Original QSCI", original_qsci.energy),
    ("SBD-TPB", sbd_result.energy),
    ("FCI", molecule.fci_energy),
]

print(f"{'method':<16} {'energy':>18} {'delta to FCI':>16}")
for label, value in rows:
    print(f"{label:<16} {value:>18.12f} {value - molecule.fci_energy:>16.6e}")

method                       energy     delta to FCI
VQE LBFGS           -4.274646013830     3.292559e-02
Original QSCI       -4.289724916941     1.784669e-02
SBD-TPB             -4.301663817964     5.907784e-03
FCI                 -4.307571601999     0.000000e+00


## References

[1] "Quantum-selected configuration interaction: classical diagonalization of hamiltonians in subspaces selected by quantum computers" K. Kanno, M. Kohda, R. Imai, S. Koh, K. Mitarai, W. Mizukami, and Y. O. Nakagawa (2023) arXiv: 2302.11320

[2] "GPU-Accelerated Selected Basis Diagonalization with Thrust for SQD-based Algorithms" J. Doi, T. Shirakawa, Y. Kawashima, S. Yunoki, and H. Horii (2026) arXiv: 2601.16637